# 04. Data Cleaning, Missing Values & Type Hygiene: Beginner Guide

### 🌟 What is Data Cleaning, Missing Values & Type Hygiene?
Real-world data is often messy, containing missing values (`NaN`), duplicates, and incorrect data types. This notebook covers identifying nulls (`isna`), imputing or removing missing records (`fillna`, `dropna`), deduplicating rows, and ensuring data type integrity.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Null Detection & Handling**: Covers `.isnull()`/`.isna()`, `.notnull()`/`.notna()`, `.dropna()`, and `.fillna()`.
- **Deduplication**: Covers `.duplicated()` and `.drop_duplicates()`.
- **Type Hygiene & Conversion**: Covers `.astype()`, `pd.to_numeric()`, and `pd.to_datetime()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount   card_type  \
0       TX110686      C82845       M2697             1216.33        Visa   
1       TX107170      C85674       M3868              324.99  MasterCard   

  transaction_status device_type  account_age_months transaction_date region  \
0             Failed         POS                  56       07-06-2025  North   
1           Reversed     Desktop                 112       10/05/2025   East   

   is_fraud  
0         0  
1         0  


### 🔹 Detecting Nulls: `.isnull()` / `.isna()`
Counts missing entries across all columns in raw_transactions.csv. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.isnull().sum()`


In [2]:
print('Missing Values Count per Column:\n', df.isnull().sum())

Missing Values Count per Column:
 transaction_id          0
customer_id             0
merchant_id             0
transaction_amount    738
card_type               0
transaction_status      0
device_type             0
account_age_months      0
transaction_date        0
region                  0
is_fraud                0
dtype: int64


### 🔹 Detecting Non-Nulls: `.notnull()` / `.notna()`
Filters rows with complete customer IDs. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df[df['customer_id'].notnull()]`


In [3]:
print('Valid Customer Rows Count:', df['customer_id'].notnull().sum())

Valid Customer Rows Count: 15000


### 🔹 Dropping Missing Values: `.dropna()`
Removes rows missing transaction amounts or dates. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.dropna(subset=['transaction_amount', 'transaction_date'])`


In [4]:
cleaned_df = df.dropna(subset=['transaction_amount'])
print('Rows after dropna on amount:', len(cleaned_df))

Rows after dropna on amount: 14262


### 🔹 Imputing Missing Values: `.fillna()`
Imputes missing numeric transaction amounts with the median amount. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df['transaction_amount'].fillna(df['transaction_amount'].median())`


In [5]:
imputed_amt = df['transaction_amount'].fillna(df['transaction_amount'].median())
print('Null count after median imputation:', imputed_amt.isnull().sum())

Null count after median imputation: 0


### 🔹 Detecting Duplicates: `.duplicated()`
Identifies duplicate transaction records. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.duplicated(subset=['transaction_id'])`


In [6]:
duplicate_count = df.duplicated(subset=['transaction_id']).sum()
print('Duplicate Transaction IDs Count:', duplicate_count)

Duplicate Transaction IDs Count: 100


### 🔹 Removing Duplicates: `.drop_duplicates()`
Deduplicates raw transactions by transaction_id. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.drop_duplicates(subset=['transaction_id'], keep='first')`


In [7]:
deduped_df = df.drop_duplicates(subset=['transaction_id'], keep='first')
print(f'Original Rows: {len(df)} -> Deduplicated Rows: {len(deduped_df)}')

Original Rows: 15000 -> Deduplicated Rows: 14900


### 🔹 Type Casting with `.astype()`
Downcasts `is_fraud` from int64 to int8 and `transaction_amount` to float32. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `df.astype({'is_fraud': 'int8', 'transaction_amount': 'float32'})`


In [8]:
optimized_df = deduped_df.astype({'is_fraud': 'int8', 'transaction_amount': 'float32'})
print('Optimized Dtypes:\n', optimized_df.dtypes[['transaction_amount', 'is_fraud']])

Optimized Dtypes:
 transaction_amount    float32
is_fraud                 int8
dtype: object


### 🔹 Robust Numeric Parsing: `pd.to_numeric()`
Coerces corrupt strings to NaN safely. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `pd.to_numeric(df['transaction_amount'], errors='coerce')`


In [9]:
coerced_nums = pd.to_numeric(df['transaction_amount'], errors='coerce')
print('Parsed Numeric Count:', coerced_nums.count())

Parsed Numeric Count: 14262


### 🔹 Robust Datetime Parsing: `pd.to_datetime()`
Parses mixed string timestamps in raw_transactions.csv into `datetime64[ns]`. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Ensure your column is converted to datetime with `pd.to_datetime()` before calling `.dt` properties like `.dt.year` or `.dt.day_name()`.

**Syntax:** `pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')`


In [10]:
clean_dates = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print('Parsed Datetime Series Head:\n', clean_dates.head())

Parsed Datetime Series Head:
 0   2025-07-06 00:00:00
1   2025-10-05 00:00:00
2   2025-07-26 06:22:30
3   2025-09-06 00:00:00
4   2026-04-17 00:00:00
Name: transaction_date, dtype: datetime64[ns]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: End-to-End Data Cleaning Pipeline

**Approach:** Build an end-to-end cleaning pipeline on raw_transactions.csv.
**Syntax:** `df.drop_duplicates().dropna().assign(...)`


In [11]:
pipeline_df = (
    df
    .drop_duplicates(subset=['transaction_id'])
    .assign(
        transaction_date=lambda d: pd.to_datetime(d['transaction_date'], format='mixed', errors='coerce'),
        transaction_amount=lambda d: pd.to_numeric(d['transaction_amount'], errors='coerce')
    )
    .dropna(subset=['transaction_amount', 'transaction_date'])
)
print(f'Cleaned Dataset Ready for Modeling: {len(pipeline_df)} valid transactions')

Cleaned Dataset Ready for Modeling: 14168 valid transactions
